# 02 — Data Cleaning

> **Food Delivery Analytics Project**  
> Applies the full data-cleaning pipeline, shows before/after comparisons,
> documents decisions, and saves the processed dataset.

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_cleaning import (
    load_raw, fix_dtypes, remove_duplicates, handle_missing,
    validate_ratings, validate_amounts, validate_delivery_times,
    cap_outliers, engineer_features, run_cleaning_pipeline
)

pd.set_option('display.max_columns', 50)
plt.rcParams.update({'figure.figsize': (10, 4), 'figure.dpi': 100})
sns.set_theme(style='whitegrid')
print('Ready')

## Step 1: Load Raw Data

In [ ]:
raw = load_raw('../data/raw/food_delivery_orders.csv')
print('Before cleaning:')
print(f'  Shape: {raw.shape}')
print(f'  Nulls: {raw.isnull().sum().sum()}')
print(f'  Duplicates: {raw.duplicated(subset="order_id").sum()}')

## Step 2: Fix Data Types

In [ ]:
df = fix_dtypes(raw.copy())
# Show before/after for a key column
print('order_date dtype before:', raw['order_date'].dtype)
print('order_date dtype after :', df['order_date'].dtype)
print('customer_rating dtype  :', df['customer_rating'].dtype)

## Step 3: Remove Duplicates

In [ ]:
before_rows = len(df)
df = remove_duplicates(df)
after_rows = len(df)
print(f'Removed {before_rows - after_rows} duplicate rows')
print(f'Remaining: {after_rows:,} rows')

## Step 4: Handle Missing Values

**Decision log:**
| Column | Strategy | Rationale |
|---|---|---|
| `customer_rating` | Keep NaN | NaN = no rating (cancelled or not rated) — meaningful absence |
| `delivery_partner_id` | Fill 'UNKNOWN' | Cannot impute; marks data gap |
| `distance_km` | Impute median by city | Distance is needed for analysis; city-level median is a reasonable proxy |
| `order_date` | Drop row | Cannot infer; less than 0.01% of rows |

In [ ]:
before_nulls = df.isnull().sum().sum()
df = handle_missing(df)
after_nulls = df.isnull().sum().sum()
print(f'Missing values before: {before_nulls}')
print(f'Missing values after : {after_nulls}')
print('\nRemaining nulls by column:')
print(df.isnull().sum()[df.isnull().sum() > 0])

## Step 5: Validate Ratings & Amounts

In [ ]:
df = validate_ratings(df)
df = validate_amounts(df)
df = validate_delivery_times(df)
print('Validation complete')
print('Rating range:', df['customer_rating'].min(), '–', df['customer_rating'].max())
print('Order amount range:', df['order_amount'].min(), '–', df['order_amount'].max())

## Step 6: Outlier Detection & Capping

We use the **IQR method** with a factor of 3.0 (more conservative than 1.5 to keep natural variation).

Values beyond `Q1 – 3×IQR` or `Q3 + 3×IQR` are **capped** (Winsorized), not dropped.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['order_amount', 'delivery_time_minutes', 'distance_km']):
    df[col].dropna().plot(kind='box', ax=ax, color='#1a3a5c')
    ax.set_title(f'{col} (before capping)')
plt.suptitle('Outlier Distributions Before Capping')
plt.tight_layout()
plt.show()

In [ ]:
df = cap_outliers(df, cols=['order_amount', 'final_amount', 'delivery_time_minutes', 'distance_km'], factor=3.0)
print('Outlier capping applied')

## Step 7: Feature Engineering

In [ ]:
df = engineer_features(df)
new_cols = ['order_month', 'order_year', 'day_of_week', 'order_hour', 'month_name',
            'day_name', 'is_weekend', 'net_revenue', 'delivery_delay',
            'delivery_performance', 'rating_category', 'time_of_day']
print('New feature columns:')
df[new_cols].head(3)

## Step 8: Save Processed Dataset

In [ ]:
df = df.sort_values('order_date').reset_index(drop=True)
df.to_csv('../data/processed/cleaned_food_delivery_orders.csv', index=False)
print(f'Saved: {df.shape[0]:,} rows × {df.shape[1]} columns')
print('Columns:', list(df.columns))

## Summary

| Step | Action | Result |
|---|---|---|
| Duplicates | Removed exact duplicate order_ids | ~31 rows removed |
| Missing values | Imputed / flagged / dropped | < 0.01% rows dropped |
| Ratings | Clamped to [1, 5] | No invalid values found |
| Amounts | Dropped zero/negative | No invalid values found |
| Outliers | IQR capping (factor=3) | ~30 values capped |
| Features | 15 new derived columns | Ready for EDA |

**Next step:** `03_exploratory_data_analysis.ipynb`